In [30]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.Chem import Lipinski
from rdkit.Chem import rdMolDescriptors
from rdkit.Chem import Crippen
from rdkit.Chem import rdFingerprintGenerator

In [31]:
df=pd.read_csv("CHEMBL230_RAW_DATA.csv")
print(df.head())

  Molecule ChEMBL ID                                             Smiles  \
0        CHEMBL70104    CC1(C)OC(=O)C(OC2CCCCC2)=C1c1ccc(S(C)(=O)=O)cc1   
1        CHEMBL96098     CCc1ccc(-c2ncc(Cl)cc2-c2ccc(S(C)(=O)=O)cc2)cn1   
2        CHEMBL97179  CCCCOC(=O)Cc1c(C)n(C(=O)c2ccc(Cl)cc2)c2ccc(OC)...   
3            CHEMBL6     COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1   
4       CHEMBL355781  CS(=O)(=O)c1ccc(-c2csc(CC(=O)O)c2-c2ccc(F)cc2)cc1   

  Standard Type  Standard Value Standard Units  
0          IC50            40.0             nM  
1          IC50          1700.0             nM  
2          IC50            50.0             nM  
3          IC50           200.0             nM  
4          IC50         10000.0             nM  


In [32]:
print("Column names :",df.columns)
print("Shape : ",df.shape)
print("Missing values :")
print(df.isnull().sum())
print("Duplicate rows:",df.duplicated().sum())
print(df["Standard Type"].value_counts(dropna=False))
print(df["Standard Units"].value_counts(dropna=False))

Column names : Index(['Molecule ChEMBL ID', 'Smiles', 'Standard Type', 'Standard Value',
       'Standard Units'],
      dtype='object')
Shape :  (7979, 5)
Missing values :
Molecule ChEMBL ID       0
Smiles                 190
Standard Type          157
Standard Value        1139
Standard Units        1120
dtype: int64
Duplicate rows: 602
Standard Type
IC50    7822
NaN      157
Name: count, dtype: int64
Standard Units
nM         6832
NaN        1120
ug.mL-1      25
ug            1
%             1
Name: count, dtype: int64


In [33]:
# Check number of rows before removing duplicates
print("Rows before removing duplicates:", len(df))

# Remove exact duplicate rows
df = df.drop_duplicates().reset_index(drop=True)

print("Rows after removing duplicates:", len(df))

print("Duplicates removed:", 7979 - len(df))

Rows before removing duplicates: 7979
Rows after removing duplicates: 7377
Duplicates removed: 602


In [34]:
required_columns = [
    "Smiles",
    "Standard Type",
    "Standard Value",
    "Standard Units"
]
print("Before removing null values:")
print("Shape :",df.shape)
print(df.isnull().sum())
df = df.dropna(subset=required_columns).reset_index(drop=True)

print("Shape after removing missing essential values:")
print("Shape",df.shape)

print("\nMissing values:")
print(df[required_columns].isnull().sum())

Before removing null values:
Shape : (7377, 5)
Molecule ChEMBL ID       0
Smiles                 167
Standard Type          134
Standard Value        1103
Standard Units        1084
dtype: int64
Shape after removing missing essential values:
Shape (6273, 5)

Missing values:
Smiles            0
Standard Type     0
Standard Value    0
Standard Units    0
dtype: int64


In [ ]:
def check_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        return mol is not None
    except:
        return False
df["Valid_SMILES"] = df["Smiles"].apply(check_smiles)

print(df["Valid_SMILES"].value_counts())

Valid_SMILES
True    6273
Name: count, dtype: int64


Index(['Molecule ChEMBL ID', 'Smiles', 'Standard Type', 'Standard Value',
       'Standard Units', 'Valid_SMILES'],
      dtype='object')

In [40]:
d=pd.read_csv("CHEMBL230_Preprocessed_Data.csv")
d.columns

Index(['molecule_chembl_id', 'smiles', 'standard_value', 'bioactivity_class'], dtype='object')

In [43]:
df["Molecule"] = df["Smiles"].apply(
    Chem.MolFromSmiles
)
df["MolWt"] = df["Molecule"].apply(
    Descriptors.MolWt
)


In [41]:
def convert_to_nM(value, unit, mol_weight):
    
    unit = str(unit).strip().lower()
    
    # Molar units
    if unit == "nm":
        return value
    
    elif unit in ["um", "µm"]:
        return value * 1000
    
    elif unit == "mm":
        return value * 1_000_000
    
    elif unit == "pm":
        return value / 1000
    
    # Mass concentration
    elif unit in ["ug.ml-1", "ug/ml", "µg/ml"]:
        if mol_weight > 0:
            return (value * 1_000_000) / mol_weight
        else:
            return np.nan
    
    # Cannot safely convert
    else:
        return np.nan

In [44]:
df["IC50_nM"] = df.apply(
    lambda row: convert_to_nM(
        row["Standard Value"],
        row["Standard Units"],
        row["MolWt"]
    ),
    axis=1
)

In [48]:
df[df['IC50_nM'].isnull()]

,Molecule ChEMBL ID,Smiles,Standard Type,Standard Value,Standard Units,Valid_SMILES,Molecule,MolWt,IC50_nM
1039,CHEMBL479210,C=C[C@@H]1[C@@H](O)C[C@@H]2[C@]3(CC[C@]4(C)[C@...,IC50,4.7,ug,True,<rdkit.Chem.rdchem.Mol object at 0x000001B59CC...,488.709,NaN
6199,CHEMBL348436,COc1cc2oc(-c3ccc(O)cc3)cc(=O)c2c(O)c1OC,IC50,6.0,%,True,<rdkit.Chem.rdchem.Mol object at 0x000001B59CD...,314.293,NaN


In [50]:
before = len(df)

df = df.dropna(
    subset=["IC50_nM"]
).reset_index(drop=True)

after = len(df)

print("Rows before:", before)
print("Rows after:", after)
print("Rows removed:", before - after)

Rows before: 6271
Rows after: 6271
Rows removed: 0


In [51]:
print(
    "IC50 values <= 0:",
    (df["IC50_nM"] <= 0).sum()
)
df = df[df["IC50_nM"] > 0].copy()

df = df.reset_index(drop=True)

print("Final shape after IC50 cleaning:", df.shape)

IC50 values <= 0: 6
Final shape after IC50 cleaning: (6265, 9)


In [52]:
df["Canonical_SMILES"] = df["Molecule"].apply(
    lambda mol: Chem.MolToSmiles(
        mol,
        canonical=True
    )
)

In [53]:
print(
    "Duplicate molecules:",
    df["Canonical_SMILES"].duplicated().sum()
)

Duplicate molecules: 1243


In [54]:
df = (
    df.groupby("Canonical_SMILES", as_index=False)
      .agg({
          "Molecule ChEMBL ID": "first",
          "Smiles": "first",
          "Standard Type": "first",
          "Standard Value": "median",
          "Standard Units": "first",
          "MolWt": "first",
          "IC50_nM": "median"
      })
)

print("Shape after removing repeated molecules:")
print(df.shape)

Shape after removing repeated molecules:
(5022, 8)


In [55]:
ACTIVITY_THRESHOLD=10000
df["Activity"] = (
    df["IC50_nM"] <= ACTIVITY_THRESHOLD
).astype(int)

In [56]:
print(df["Activity"].value_counts())

print("\nPercentage:")
print(
    df["Activity"]
    .value_counts(normalize=True) * 100
)

Activity
1    3586
0    1436
Name: count, dtype: int64

Percentage:
Activity
1    71.405814
0    28.594186
Name: proportion, dtype: float64
